In [ ]:
import pandas as pd
from pymatgen.core.composition import Composition

In [ ]:
data = pd.read_csv('./data/data.csv')
def get_composition(composition):
    return Composition(composition).fractional_composition
data['composition'] = data['composition'].apply(get_composition)


In [ ]:
param = 'wt'
unit = 'wt%'
data = data.dropna(subset=[param])

# 特征工程

## 原子的摩尔分数

In [ ]:
elements_list = ['V', 'Cu', 'Ce', 'Zr', 'Co', 'W', 'Mn', 'Mo', 'Cr', 'Sn', 'Ni', 'Gd', 'La', 'Ti', 'Si', 'B', 'Nb', 'Y', 'Fe', 'Al', 'Mg', 'C']
for element in elements_list:
    def get_atomic_fraction(comps):
        return comps.get_atomic_fraction(element)
    data[element] = data['composition'].apply(get_atomic_fraction)


# Training model

In [ ]:
from sklearn import metrics
from sklearn.model_selection import cross_validate, train_test_split
# import scikitplot as skplt
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn import svm

from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import MinMaxScaler
import numpy as np
from sklearn.utils import shuffle
from matplotlib import pyplot as plt
from matplotlib import rcParams

rcParams['font.family']='sans-serif'
plt.rcParams['font.sans-serif'] = ['Times New Roman']  # 如果要显示中文字体,则在此处设为：SimHei
plt.rcParams['axes.unicode_minus'] = False  # 显示负号
RNG_SEED = 10

In [ ]:
data

In [ ]:
X = data.iloc[:, 7:].to_numpy()
y = data[param].to_numpy()
minmax_scaler = MinMaxScaler()
X = minmax_scaler.fit_transform(X)
X, y = shuffle(X, y, random_state=RNG_SEED)
print(X.shape, y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=RNG_SEED) 
print(X_train.shape, X_test.shape)
print(y_train.shape, y_test.shape)

# RF

In [ ]:
rf_model = RandomForestRegressor(n_estimators=50, n_jobs=-1, random_state=RNG_SEED)
rf_model.fit(X_train, y_train)

In [ ]:
print("Random Forest:")
y_pred = rf_model.predict(X_test)
print('R2: ', rf_model.score(X_test, y_test))  # sklearnrint("MSE: ", mean_squared_error(y_test, model.predict(X_test))) # mse
print("MAE: ", metrics.mean_absolute_error(y_test, y_pred))

cv = cross_validate(rf_model, X, y, cv=5, scoring=('neg_mean_absolute_error', 'r2'), n_jobs=-1)
print('test_neg_mean_absolute_error: ', np.mean(cv['test_neg_mean_absolute_error']))
print('test_r2: ', np.mean(cv['test_r2']))

plt.figure(figsize=(8, 6))
# act_pred(y_test, y_pred, x_hist=False, y_hist=False)
plt.scatter(y_test, y_pred, alpha=0.5, color='red', s=100, label='RF')
plt.xlabel(f'Actual value ({unit})', fontsize=22)
plt.ylabel('Predicted value ({unit})', fontsize=22)
# plt.legend(fontsize=18)
xlim = plt.xlim()
ylim = plt.ylim()
lims = [min(xlim[0], ylim[0]), max(xlim[1], ylim[1])]
min_val = lims[0]
max_val = lims[1]
# 确定对角线的坐标
diag_x = [min_val, max_val]
diag_y = [min_val, max_val]
plt.plot(diag_x, diag_y, linestyle='--', color='black')
plt.xlim(lims[0], lims[1])
plt.ylim(lims[0], lims[1])
plt.xticks(fontsize=18)
# 设置 y 轴刻度标签的大小
plt.yticks(fontsize=18)
plt.savefig(f"./figures/RF_{param}.jpg", dpi=600)


# NN

In [ ]:
mlp_model = MLPRegressor(hidden_layer_sizes=(50, 50, 100, 100, 50, 50), random_state=RNG_SEED)
mlp_model.fit(X_train, y_train)

In [ ]:
print("MLP:")
y_pred = mlp_model.predict(X_test)
print('R2: ', mlp_model.score(X_test, y_test))  # sklearnrint("MSE: ", mean_squared_error(y_test, model.predict(X_test))) # mse
print("MAE: ", metrics.mean_absolute_error(y_test, y_pred))

cv = cross_validate(mlp_model, X, y, cv=5, scoring=('neg_mean_absolute_error', 'r2'), n_jobs=-1)
print('test_neg_mean_absolute_error: ', np.mean(cv['test_neg_mean_absolute_error']))
print('test_r2: ', np.mean(cv['test_r2']))

plt.figure(figsize=(8, 6))
# act_pred(y_test, y_pred, x_hist=False, y_hist=False)
plt.scatter(y_test, y_pred, alpha=0.5, color='green', s=100, label='MLP')
plt.xlabel(f'Actual value ({unit})', fontsize=22)
plt.ylabel(f'Predicted value ({unit})', fontsize=22)
xlim = plt.xlim()
ylim = plt.ylim()
lims = [min(xlim[0], ylim[0]), max(xlim[1], ylim[1])]
min_val = lims[0]
max_val = lims[1]
# 确定对角线的坐标
diag_x = [min_val, max_val]
diag_y = [min_val, max_val]
plt.plot(diag_x, diag_y, linestyle='--', color='black')
plt.xlim(lims[0], lims[1])
plt.ylim(lims[0], lims[1])
plt.xticks(fontsize=18)
# 设置 y 轴刻度标签的大小
plt.yticks(fontsize=18)
plt.savefig(f"./figures/MLP_{param}.jpg", dpi=600)

# ET

In [ ]:
et_model = ExtraTreesRegressor(n_estimators=50, random_state=RNG_SEED)
et_model.fit(X_train, y_train)

In [ ]:
print("ET:")
y_pred = et_model.predict(X_test)
print('R2: ', et_model.score(X_test, y_test))  # sklearnrint("MSE: ", mean_squared_error(y_test, model.predict(X_test))) # mse
print("MAE: ", metrics.mean_absolute_error(y_test, y_pred))

cv = cross_validate(et_model, X, y, cv=5, scoring=('neg_mean_absolute_error', 'r2'), n_jobs=-1)
print('test_neg_mean_absolute_error: ', np.mean(cv['test_neg_mean_absolute_error']))
print('test_r2: ', np.mean(cv['test_r2']))

plt.figure(figsize=(8, 6))
# act_pred(y_test, y_pred, x_hist=False, y_hist=False)
plt.scatter(y_test, y_pred, alpha=0.5, color='#F26800', s=100, label='ET')
plt.xlabel(f'Actual value ({unit})', fontsize=22)
plt.ylabel(f'Predicted value ({unit})', fontsize=22)

xlim = plt.xlim()
ylim = plt.ylim()
lims = [min(xlim[0], ylim[0]), max(xlim[1], ylim[1])]
min_val = lims[0]
max_val = lims[1]
# 确定对角线的坐标
diag_x = [min_val, max_val]
diag_y = [min_val, max_val]
plt.plot(diag_x, diag_y, linestyle='--', color='black')
plt.xlim(lims[0], lims[1])
plt.ylim(lims[0], lims[1])
plt.xticks(fontsize=18)
# 设置 y 轴刻度标签的大小
plt.yticks(fontsize=18)
plt.savefig(f"./figures/ET_{param}.jpg", dpi=600)

# SVM

In [ ]:
rbf_svr_model = svm.SVR(kernel='rbf', C=100)
rbf_svr_model.fit(X_train, y_train)

In [ ]:
print("SVM:")
y_pred = rbf_svr_model.predict(X_test)
print('R2: ', rbf_svr_model.score(X_test, y_test))  # sklearnrint("MSE: ", mean_squared_error(y_test, model.predict(X_test))) # mse
print("MAE: ", metrics.mean_absolute_error(y_test, y_pred))

cv = cross_validate(rbf_svr_model, X, y, cv=5, scoring=('neg_mean_absolute_error', 'r2'), n_jobs=-1)
print('test_neg_mean_absolute_error: ', np.mean(cv['test_neg_mean_absolute_error']))
print('test_r2: ', np.mean(cv['test_r2']))

plt.figure(figsize=(8, 6))
# act_pred(y_test, y_pred, x_hist=False, y_hist=False)
plt.scatter(y_test, y_pred, alpha=0.5, color='#7A7A7A', s=100, label='SVM')
plt.xlabel(f'Actual value ({unit})', fontsize=22)
plt.ylabel(f'Predicted value ({unit})', fontsize=22)

xlim = plt.xlim()
ylim = plt.ylim()
lims = [min(xlim[0], ylim[0]), max(xlim[1], ylim[1])]
min_val = lims[0]
max_val = lims[1]
# 确定对角线的坐标
diag_x = [min_val, max_val]
diag_y = [min_val, max_val]
plt.plot(diag_x, diag_y, linestyle='--', color='black')
plt.xlim(lims[0], lims[1])
plt.ylim(lims[0], lims[1])
plt.xticks(fontsize=18)
# 设置 y 轴刻度标签的大小
plt.yticks(fontsize=18)
plt.savefig(f"./figures/SVM_{param}.jpg", dpi=600)

# 存储模型

In [ ]:
X = data.iloc[:, 7:].to_numpy()
y = data[param].to_numpy()
minmax_scaler = MinMaxScaler()
X = minmax_scaler.fit_transform(X)
et_model = RandomForestRegressor(n_estimators=50, random_state=RNG_SEED)
# et_model = ExtraTreesRegressor(n_estimators=50, random_state=RNG_SEED)
et_model.fit(X, y)

In [ ]:
import pickle
with open(f"./model/minmax_scaler_{param}.pickle", "wb") as file:
    pickle.dump(minmax_scaler, file)
with open(f"./model/{param}_model.pickle", "wb") as file:
    pickle.dump(et_model, file)